# Phase 5 — Deployment

This notebook wires together the fine-tuned ViT-B/16 + ArcFace model (Phase 3)  
with the FAISS indexes (Phase 4) and packages everything into a runnable  
Streamlit app ready for local use and HuggingFace Spaces deployment.

**Cells at a glance**
1. Imports & config  
2. Load model from checkpoint  
3. Load FAISS HNSW index + catalog metadata  
4. `search_by_image` function  
5. Pipeline smoke-test (latency check)  
6. Launch Streamlit app  
7. HuggingFace Spaces deployment guide  

## Cell 1 — Imports & Config

In [7]:
import os, time
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from transformers import AutoModel
import faiss

# ── Paths ──────────────────────────────────────────────────────────────────
CKPT_PATH   = r'C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt'
INDEX_DIR   = r'C:\visual_searh_project\faiss_index'

HNSW_INDEX_PATH = os.path.join(INDEX_DIR, 'sop_hnsw.index')
IVF_INDEX_PATH  = os.path.join(INDEX_DIR, 'sop_ivf.index')
FLAT_INDEX_PATH = os.path.join(INDEX_DIR, 'sop_flat.index')
PATHS_NPY       = os.path.join(INDEX_DIR, 'catalog_paths.npy')
LABELS_NPY      = os.path.join(INDEX_DIR, 'catalog_labels.npy')

# ── Model constants ────────────────────────────────────────────────────────
MODEL_NAME = 'google/vit-base-patch16-224-in21k'
EMBED_DIM  = 512

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'FAISS  : {faiss.__version__}')

# ── Image transform (must match training) ─────────────────────────────────
TEST_TF = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print('Config ready.')

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM   : 4.3 GB
FAISS  : 1.14.1
Config ready.


## Cell 2 — Load Model

In [8]:
class ViTEmbedder(nn.Module):
    """ViT-B/16 backbone with a two-layer projection head that outputs
    L2-normalised 512-d embeddings. Architecture must match Phase 3 exactly."""

    def __init__(self, model_name=MODEL_NAME, embed_dim=EMBED_DIM):
        super().__init__()
        self.vit = AutoModel.from_pretrained(model_name)
        self.proj = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(),
            nn.Linear(768, embed_dim),
        )

    def forward(self, x):
        out = self.vit(pixel_values=x).last_hidden_state[:, 0, :]
        return nn.functional.normalize(self.proj(out), p=2, dim=1)


def load_model(ckpt_path: str, device: torch.device) -> ViTEmbedder:
    """Instantiate ViTEmbedder and load weights from checkpoint."""
    print(f'Loading checkpoint: {ckpt_path}')
    ck = torch.load(ckpt_path, map_location=device)

    model = ViTEmbedder().to(device)
    model.vit.load_state_dict(ck['vit_state'])
    model.proj.load_state_dict(ck['proj_state'])
    model.eval()

    epoch   = ck.get('epoch',    '?')
    recall  = ck.get('recall10', None)
    r_str   = f'{recall * 100:.2f}%' if recall is not None else 'N/A'
    print(f'Loaded epoch {epoch} | val Recall@10 = {r_str}')
    print('Model ready.')
    return model


model = load_model(CKPT_PATH, DEVICE)

Loading checkpoint: C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loaded epoch 10 | val Recall@10 = 84.06%
Model ready.


## Cell 3 — Load FAISS Index + Catalog Metadata

In [9]:
def load_index_and_catalog(
    index_path: str,
    paths_npy:  str,
    labels_npy: str,
):
    """Load a FAISS index from disk together with catalog paths and labels."""
    print(f'Loading FAISS index : {index_path}')
    index = faiss.read_index(index_path)
    print(f'  Vectors in index  : {index.ntotal:,}')

    catalog_paths  = np.load(paths_npy,  allow_pickle=True)
    catalog_labels = np.load(labels_npy, allow_pickle=True)
    print(f'  Catalog paths     : {len(catalog_paths):,}')
    print(f'  Catalog labels    : {len(catalog_labels):,}')
    return index, catalog_paths, catalog_labels


# Default to HNSW (best speed/accuracy trade-off for single queries)
index, catalog_paths, catalog_labels = load_index_and_catalog(
    HNSW_INDEX_PATH, PATHS_NPY, LABELS_NPY
)
print('Index and catalog loaded.')

Loading FAISS index : C:\visual_searh_project\faiss_index\sop_hnsw.index
  Vectors in index  : 120,053
  Catalog paths     : 120,053
  Catalog labels    : 120,053
Index and catalog loaded.


## Cell 4 — `search_by_image` Function

Accepts either a **file path** (string) or a **PIL Image** object so the same
function works from both this notebook and the Streamlit app.

In [10]:
def search_by_image(
    image_path_or_pil,
    model,
    index,
    catalog_paths,
    catalog_labels,
    transform,
    device,
    k: int = 5,
):
    """
    Run a visual similarity search against the FAISS index.

    Parameters
    ----------
    image_path_or_pil : str | PIL.Image.Image
        Query image — either an absolute file path or an already-opened PIL image.
    model             : ViTEmbedder
        Fine-tuned model in eval mode.
    index             : faiss.Index
        FAISS index containing the catalog embeddings.
    catalog_paths     : np.ndarray
        Absolute paths for every item in the catalog (shape: N,).
    catalog_labels    : np.ndarray
        Class IDs for every catalog item (shape: N,).
    transform         : torchvision.transforms.Compose
        Preprocessing pipeline that matches the model's training transform.
    device            : torch.device
    k                 : int
        Number of nearest neighbours to return.

    Returns
    -------
    results    : list[dict]
        Each dict has keys: rank (int), path (str), label (int), similarity (float).
    latency_ms : float
        End-to-end wall-clock time in milliseconds (image load + encode + search).
    """
    t0 = time.perf_counter()

    # ── 1. Load image ──────────────────────────────────────────────────────
    if isinstance(image_path_or_pil, (str, os.PathLike)):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        # Assume PIL Image
        img = image_path_or_pil.convert('RGB')

    # ── 2. Preprocess & encode ─────────────────────────────────────────────
    img_t = transform(img).unsqueeze(0).to(device)  # (1, 3, 224, 224)

    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            query_emb = model(img_t).cpu().numpy().astype('float32')  # (1, 512)

    # ── 3. FAISS search ────────────────────────────────────────────────────
    sims, idxs = index.search(query_emb, k)  # sims/idxs: (1, k)

    latency_ms = (time.perf_counter() - t0) * 1000

    # ── 4. Build results list ──────────────────────────────────────────────
    results = []
    for rank in range(k):
        ci = int(idxs[0][rank])
        results.append({
            'rank':       rank + 1,
            'path':       str(catalog_paths[ci]),
            'label':      int(catalog_labels[ci]),
            'similarity': float(sims[0][rank]),
        })

    return results, latency_ms


print('search_by_image defined.')

search_by_image defined.


## Cell 5 — Test the Pipeline

Pick a random image from the catalog and run the full pipeline.  
Assert that latency is under 300 ms.

In [11]:
import random

# Pick a random catalog image that actually exists on disk
rng = random.Random(42)
sample_path = None
for candidate in rng.sample(list(catalog_paths), min(50, len(catalog_paths))):
    if os.path.isfile(str(candidate)):
        sample_path = str(candidate)
        break

if sample_path is None:
    print('WARNING: Could not find any catalog image on disk. '
          'Run this cell from a machine where the SOP dataset is mounted.')
else:
    print(f'Query image : {sample_path}')

    results, latency = search_by_image(
        image_path_or_pil=sample_path,
        model=model,
        index=index,
        catalog_paths=catalog_paths,
        catalog_labels=catalog_labels,
        transform=TEST_TF,
        device=DEVICE,
        k=5,
    )

    print(f'Latency     : {latency:.2f} ms')
    print()
    print(f'{"Rank":<6} {"Label":<10} {"Similarity":<12} {"Filename"}')
    print('-' * 70)
    for r in results:
        print(f'{r["rank"]:<6} {r["label"]:<10} {r["similarity"]:<12.4f} '
              f'{os.path.basename(r["path"])}')

    # ── Latency assertion ──────────────────────────────────────────────────
    LATENCY_BUDGET_MS = 300
    if latency < LATENCY_BUDGET_MS:
        print(f'\n✓ Latency {latency:.1f} ms < {LATENCY_BUDGET_MS} ms budget — PASS')
    else:
        print(f'\n✗ Latency {latency:.1f} ms >= {LATENCY_BUDGET_MS} ms budget — FAIL '
              '(consider using HNSW or IVF with lower nprobe)')

Query image : C:\visual_searh_project\data\raw_data\Stanford_Online_Products\Stanford_Online_Products\fan_final/400790374936_1.JPG
Latency     : 63.04 ms

Rank   Label      Similarity   Filename
----------------------------------------------------------------------
1      15436      1.0000       400790374936_1.JPG
2      3860       0.5960       151781475928_0.JPG
3      15220      0.5017       301711449291_2.JPG
4      15436      0.4953       400790374936_0.JPG
5      15220      0.4472       301711449291_0.JPG

✓ Latency 63.0 ms < 300 ms budget — PASS


## Cell 6 — Launch Streamlit App

The app file lives at `C:\visual_searh_project\app.py`.  
Run the command below in a terminal (not in this notebook).  
The cell just prints the instructions and the exact command to copy-paste.

In [12]:
STREAMLIT_APP = r'C:\visual_searh_project\app.py'

print('=' * 60)
print('  HOW TO RUN THE STREAMLIT APP')
print('=' * 60)
print()
print('1. Open a new terminal (Anaconda Prompt, PowerShell, or CMD).')
print('2. Activate your environment, e.g.:')
print('     conda activate <your_env>')
print()
print('3. Run:')
print(f'     streamlit run "{STREAMLIT_APP}"')
print()
print('4. Streamlit will open http://localhost:8501 in your browser.')
print()
print('Tip: keep this notebook running — the browser tab will auto-refresh.')
print('=' * 60)

# Optionally launch from within the notebook (uncomment if desired)
# import subprocess, sys
# subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', STREAMLIT_APP])
# print('Streamlit process launched in background.')

  HOW TO RUN THE STREAMLIT APP

1. Open a new terminal (Anaconda Prompt, PowerShell, or CMD).
2. Activate your environment, e.g.:
     conda activate <your_env>

3. Run:
     streamlit run "C:\visual_searh_project\app.py"

4. Streamlit will open http://localhost:8501 in your browser.

Tip: keep this notebook running — the browser tab will auto-refresh.


## Cell 7 — HuggingFace Spaces Deployment

HF Spaces runs the app from a Git repo. The `hf_space/` directory
(created as part of Phase 5) contains everything that needs to be pushed.

In [13]:
HF_SPACE_DIR = r'C:\visual_searh_project\hf_space'

required_files = [
    ('hf_space/app.py',           'Streamlit app (HF-adapted, loads model from HF Hub)'),
    ('hf_space/requirements.txt', 'Pinned Python dependencies'),
    ('hf_space/README.md',        'Space card with YAML frontmatter (sdk: streamlit)'),
]

large_files_to_upload_separately = [
    ('checkpoints/vit_sop_arcface_best.pt',   '~340 MB — upload via HF Hub API or git-lfs'),
    ('faiss_index/sop_hnsw.index',             '~240 MB — upload via HF Hub API or git-lfs'),
    ('faiss_index/catalog_labels.npy',         '~480 KB'),
    ('faiss_index/catalog_paths.npy',          '~  ? MB — OPTIONAL; needed only for thumbnail display'),
]

print('=' * 64)
print('  HUGGINGFACE SPACES DEPLOYMENT GUIDE')
print('=' * 64)
print()
print('STEP 1  Create a new Space at https://huggingface.co/new-space')
print('        SDK   → Streamlit')
print('        Visibility → Public (or Private)')
print()
print('STEP 2  Clone the empty Space repo:')
print('        git clone https://huggingface.co/spaces/<your_username>/<space_name>')
print()
print('STEP 3  Copy the following files from hf_space/ into the cloned repo root:')
print()
for fname, desc in required_files:
    status = '✓ exists' if os.path.isfile(os.path.join(r'C:\visual_searh_project', fname)) else '  create'
    print(f'  {fname:<35} — {desc}')
print()
print('STEP 4  Upload large binary files with git-lfs (already enabled on HF):')
print()
for fname, note in large_files_to_upload_separately:
    print(f'  {fname:<45} {note}')
print()
print('STEP 5  Commit & push:')
print('        git add .')
print('        git commit -m "Initial deployment"')
print('        git push')
print()
print('STEP 6  Monitor build logs in the Space\'s "Logs" tab.')
print('        The app starts automatically once the build succeeds.')
print()
print('NOTE    The HF Space app (hf_space/app.py) will display a clear error')
print('        message if the checkpoint is not found, rather than crashing.')
print('=' * 64)

# Verify required HF Space files exist on disk
print()
print('Local file check:')
all_ok = True
for fname, _ in required_files:
    full = os.path.join(r'C:\visual_searh_project', fname)
    exists = os.path.isfile(full)
    mark = '✓' if exists else '✗ MISSING'
    print(f'  {mark}  {full}')
    if not exists:
        all_ok = False
if all_ok:
    print()
    print('All required HF Space files are present. Ready to push!')
else:
    print()
    print('Some files are missing. Re-run the Phase 5 file creation cells.')

  HUGGINGFACE SPACES DEPLOYMENT GUIDE

STEP 1  Create a new Space at https://huggingface.co/new-space
        SDK   → Streamlit
        Visibility → Public (or Private)

STEP 2  Clone the empty Space repo:
        git clone https://huggingface.co/spaces/<your_username>/<space_name>

STEP 3  Copy the following files from hf_space/ into the cloned repo root:

  hf_space/app.py                     — Streamlit app (HF-adapted, loads model from HF Hub)
  hf_space/requirements.txt           — Pinned Python dependencies
  hf_space/README.md                  — Space card with YAML frontmatter (sdk: streamlit)

STEP 4  Upload large binary files with git-lfs (already enabled on HF):

  checkpoints/vit_sop_arcface_best.pt           ~340 MB — upload via HF Hub API or git-lfs
  faiss_index/sop_hnsw.index                    ~240 MB — upload via HF Hub API or git-lfs
  faiss_index/catalog_labels.npy                ~480 KB
  faiss_index/catalog_paths.npy                 ~  ? MB — OPTIONAL; needed only